In [8]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
from sklearn.model_selection import train_test_split

In [9]:
texts = [
    "I love this product",
    "This is a bad movie",
    "Excellent book",
    "Worst experience ever",
    "I feel great",
    "I hate it",
    "I hate this movie but ending is good",  # mixed sentiment
    "The movie was boring but visuals were amazing"
]

labels = [1, 0, 1, 0, 1, 0, 1, 1]  # 1 = positive, 0 = negative (label mixed sentences as positive for simplicity)

In [10]:
max_words = 1000                   # Maximum number of words to keep in vocabulary  --- Only top 1000 most frequent words will be considered
max_len = 15                       # Maximum length of each sequence (sentence) ----  All sequences will be padded/truncated to length 15

# Create tokenizer
# oov_token = "<OOV>" will replace unknown words
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")       # num_words → limit vocab size to 1000        # oov_token → replace unknown words with "<OOV>"

# Fit tokenizer on texts
# Example:
# "students of aimlc are brilliant"
# → {'students':2, 'of':3, 'aimlc':4, 'are':5, 'brilliant':6}
tokenizer.fit_on_texts(texts)                                       # Assigns a unique integer to each word

# Convert texts to sequences
# Example:
# "students of aimlc are brilliant"
# → [2, 3, 4, 5, 6]
sequences = tokenizer.texts_to_sequences(texts)                     # Convert each sentence into sequence of integers

# Pad sequences to ensure same length (15)
# padding='post' → add zeros at the end if sentence is short
# Longer sentences will be truncated
# Apply padding
# max_len = 10
# padding='post' → add zeros at end
# Example:
# [2, 3, 4, 5, 6]
# → [2, 3, 4, 5, 6, 0, 0, 0, 0, 0]
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post')

# Print example output
# Example output:
# "students of aimlc are brilliant"
# → [2 3 4 5 6 0 0 0 0 0]
print("Example padded sequence:\n", padded_sequences[2])

Example padded sequence:
 [12 13  0  0  0  0  0  0  0  0  0  0  0  0  0]


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, labels, test_size=0.25, random_state=42               # test_size=0.25 → 25% data for testing, 75% for training.
                                                                            # random_state=42 → ensures same split every time (reproducibility)
)
# padded_sequences =
# [
#   [2, 3, 4, 0, 0],
#   [5, 6, 0, 0, 0],
#   [7, 8, 9, 0, 0],
#   [1, 2, 3, 4, 5]
# ]

# labels = [1, 0, 1, 0]
# After split (75% train, 25% test):

# X_train =
# [
#   [2, 3, 4, 0, 0],
#   [5, 6, 0, 0, 0],
#   [7, 8, 9, 0, 0]
# ]

# y_train = [1, 0, 1]

# X_test =
# [
#   [1, 2, 3, 4, 5]
# ]
# y_test = [0]


In [12]:
embedding_dim = 50                               # Define embedding dimension (size of word vector representation)

rnn_model = Sequential([

    # Embedding Layer:
    # Converts word indices into dense vectors of fixed size (50 here)
    # input_dim = vocabulary size (max_words)
    # output_dim = embedding size (50)
    # input_length = length of each input sequence
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),

    # Simple RNN Layer:
    # 32 = number of hidden units (neurons)
    # Processes sequence step-by-step and captures temporal dependencies
    SimpleRNN(32),

    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

rnn_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
rnn_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [13]:
rnn_history = rnn_model.fit(
    X_train, np.array(y_train),
    epochs=15,
    batch_size=2,
    validation_data=(X_test, np.array(y_test))
)


Epoch 1/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 143ms/step - accuracy: 0.1667 - loss: 0.7305 - val_accuracy: 0.0000e+00 - val_loss: 0.8670
Epoch 2/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.6667 - loss: 0.6571 - val_accuracy: 0.0000e+00 - val_loss: 0.9383
Epoch 3/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8333 - loss: 0.4992 - val_accuracy: 0.0000e+00 - val_loss: 1.0308
Epoch 4/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8333 - loss: 0.5127 - val_accuracy: 0.0000e+00 - val_loss: 1.0947
Epoch 5/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8333 - loss: 0.4295 - val_accuracy: 0.0000e+00 - val_loss: 1.1971
Epoch 6/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8333 - loss: 0.4558 - val_accuracy: 0.0000e+00 - val_loss: 1.2911
Epoch 7/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.8333 - loss: 0.2921 - val_accuracy: 0.0000e+00 - val_loss: 1.3653
Epoch 8/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.8333 - loss: 0.3249 - val_accurac

In [14]:
new_text = ["I hate this movie but the ending is very bad"]

seq = tokenizer.texts_to_sequences(new_text)
padded_seq = pad_sequences(seq, maxlen=max_len, padding='post')
pred = rnn_model.predict(padded_seq)

print(f"Sentiment score: {pred[0][0]:.2f}")  # >0.5 → positive, <0.5 → negative



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step
Sentiment score: 0.95


LSTM


In [17]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split

In [18]:
texts = [
    "I love this product",
    "This is a bad movie",
    "Excellent book",
    "Worst experience ever",
    "I feel great",
    "I hate it",
    "I hate this movie but ending is good",  # mixed sentiment
    "The movie was boring but visuals were amazing"
]

labels = [1, 0, 1, 0, 1, 0, 1, 1]  # 1 = positive, 0 = negative

In [19]:
max_words = 1000
max_len = 15

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post')

print("Example padded sequence:\n", padded_sequences[6])

Example padded sequence:
 [ 2  6  3  4  7 20  5 21  0  0  0  0  0  0  0]


In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, labels, test_size=0.25, random_state=42
)

In [21]:
embedding_dim = 50

lstm_model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
    LSTM(64),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # binary classification
])

lstm_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [22]:
lstm_history = lstm_model.fit(
    X_train, np.array(y_train),
    epochs=20,
    batch_size=2,
    validation_data=(X_test, np.array(y_test))
)

Epoch 1/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 142ms/step - accuracy: 0.3333 - loss: 0.6977 - val_accuracy: 0.0000e+00 - val_loss: 0.7245
Epoch 2/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.8333 - loss: 0.6734 - val_accuracy: 0.0000e+00 - val_loss: 0.7625
Epoch 3/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.8333 - loss: 0.6555 - val_accuracy: 0.0000e+00 - val_loss: 0.8199
Epoch 4/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8333 - loss: 0.6097 - val_accuracy: 0.0000e+00 - val_loss: 0.8881
Epoch 5/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8333 - loss: 0.5799 - val_accuracy: 0.0000e+00 - val_loss: 0.9784
Epoch 6/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.8333 - loss: 0.5557 - val_accuracy: 0.0000e+00 - val_loss: 1.1033
Epoch 7/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.8333 - loss: 0.5654 - val_accuracy: 0.0000e+00 - val_loss: 1.3315
Epoch 8/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8333 - loss: 0.5013 - val_accurac

In [23]:
loss, accuracy = lstm_model.evaluate(X_test, np.array(y_test))
#print(f"Test Accuracy: {accuracy*100:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 287ms/step - accuracy: 0.0000e+00 - loss: 2.8578


In [24]:
new_texts = [
    "I hate this movie but the ending is good",
    "Absolutely loved the story and characters"
]

seq = tokenizer.texts_to_sequences(new_texts)
padded_seq = pad_sequences(seq, maxlen=max_len, padding='post')
predictions = lstm_model.predict(padded_seq)

for text, pred in zip(new_texts, predictions):
    sentiment = "Positive" if pred[0] > 0.5 else "Negative"
    print(f"Text: {text}\nPredicted sentiment: {sentiment}\n")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step
Text: I hate this movie but the ending is good
Predicted sentiment: Positive

Text: Absolutely loved the story and characters
Predicted sentiment: Positive



GRU GATED RECURRENT UNIT


In [25]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from sklearn.model_selection import train_test_split

In [26]:
texts = [
    "I love this product",
    "This is a bad movie",
    "Excellent book",
    "Worst experience ever",
    "I feel great",
    "I hate it",
    "I hate this movie but ending is good",  # mixed sentiment
    "The movie was boring but visuals were amazing"
]

labels = [1, 0, 1, 0, 1, 0, 1, 1]  # 1 = positive, 0 = negative

In [27]:
max_words = 1000
max_len = 15

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post')

In [28]:
embedding_dim = 50

gru_model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
    GRU(64),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # binary classification
])

gru_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
gru_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [29]:
gru_history = gru_model.fit(
    X_train, np.array(y_train),
    epochs=20,
    batch_size=2,
    validation_data=(X_test, np.array(y_test))
)

Epoch 1/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 142ms/step - accuracy: 0.8333 - loss: 0.6799 - val_accuracy: 0.0000e+00 - val_loss: 0.7263
Epoch 2/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.8333 - loss: 0.6583 - val_accuracy: 0.0000e+00 - val_loss: 0.7762
Epoch 3/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8333 - loss: 0.6422 - val_accuracy: 0.0000e+00 - val_loss: 0.8331
Epoch 4/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8333 - loss: 0.6134 - val_accuracy: 0.0000e+00 - val_loss: 0.9025
Epoch 5/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.8333 - loss: 0.5769 - val_accuracy: 0.0000e+00 - val_loss: 0.9730
Epoch 6/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8333 - loss: 0.5219 - val_accuracy: 0.0000e+00 - val_loss: 1.0692
Epoch 7/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8333 - loss: 0.5027 - val_accuracy: 0.0000e+00 - val_loss: 1.2130
Epoch 8/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8333 - loss: 0.5248 - val_accurac

In [30]:
loss, accuracy = gru_model.evaluate(X_test, np.array(y_test))
#print(f"Test Accuracy: {accuracy*100:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 259ms/step - accuracy: 0.0000e+00 - loss: 1.5985


In [31]:
new_texts = [
    "I hate this movie but the ending is good",
    "Absolutely loved the story and characters"
]

seq = tokenizer.texts_to_sequences(new_texts)
padded_seq = pad_sequences(seq, maxlen=max_len, padding='post')
predictions = gru_model.predict(padded_seq)

for text, pred in zip(new_texts, predictions):
    sentiment = "Positive" if pred[0] > 0.5 else "Negative"
    print(f"Text: {text}\nPredicted sentiment: {sentiment}\n")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step
Text: I hate this movie but the ending is good
Predicted sentiment: Positive

Text: Absolutely loved the story and characters
Predicted sentiment: Positive



RNN - LSTM - GRU

In [32]:
print("\n=== FINAL COMPARISON ===")

print(f"RNN  -> Train Acc: {rnn_history.history['accuracy'][-1]:.4f}, "
      f"Loss: {rnn_history.history['loss'][-1]:.4f}, "
      f"Val Acc: {rnn_history.history['val_accuracy'][-1]:.4f}, "
      f"Val Loss: {rnn_history.history['val_loss'][-1]:.4f}")

print(f"LSTM -> Train Acc: {lstm_history.history['accuracy'][-1]:.4f}, "
      f"Loss: {lstm_history.history['loss'][-1]:.4f}, "
      f"Val Acc: {lstm_history.history['val_accuracy'][-1]:.4f}, "
      f"Val Loss: {lstm_history.history['val_loss'][-1]:.4f}")

print(f"GRU  -> Train Acc: {gru_history.history['accuracy'][-1]:.4f}, "
      f"Loss: {gru_history.history['loss'][-1]:.4f}, "
      f"Val Acc: {gru_history.history['val_accuracy'][-1]:.4f}, "
      f"Val Loss: {gru_history.history['val_loss'][-1]:.4f}")


=== FINAL COMPARISON ===
RNN  -> Train Acc: 1.0000, Loss: 0.0578, Val Acc: 0.5000, Val Loss: 1.7511
LSTM -> Train Acc: 0.8333, Loss: 0.1967, Val Acc: 0.0000, Val Loss: 2.8578
GRU  -> Train Acc: 0.8333, Loss: 0.4364, Val Acc: 0.0000, Val Loss: 1.5985


LSTM - IMDB DATASET

In [33]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout ,Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

In [34]:
max_words = 20000  # vocabulary size

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=max_words)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training samples: 25000
Testing samples: 25000


In [35]:
max_len = 300

X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

In [36]:
model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    LSTM(128),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [37]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 175s 552ms/step - accuracy: 0.7234 - loss: 0.5229 - val_accuracy: 0.8436 - val_loss: 0.3764
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 201s 550ms/step - accuracy: 0.8899 - loss: 0.2842 - val_accuracy: 0.8580 - val_loss: 0.3346
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 167s 534ms/step - accuracy: 0.9216 - loss: 0.2131 - val_accuracy: 0.8570 - val_loss: 0.3778
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 189s 493ms/step - accuracy: 0.9524 - loss: 0.1365 - val_accuracy: 0.8678 - val_loss: 0.3770
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 154s 492ms/step - accuracy: 0.9732 - loss: 0.0859 - val_accuracy: 0.8654 - val_loss: 0.4476
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 201s 490ms/step - accuracy: 0.9728 - loss: 0.0802 - val_accuracy: 0.8610 - val_loss: 0.4966
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 151s 483ms/step - accuracy: 0.9725 - loss: 0.0771 - val_accuracy: 0.8032 - val_loss: 0.7094
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 152s 486ms/step - accuracy: 0.9880 -

In [38]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy*100:.2f}%")

782/782 ━━━━━━━━━━━━━━━━━━━━ 65s 83ms/step - accuracy: 0.8371 - loss: 0.6654
Test Accuracy: 83.71%
